Accessing IPython cluster clients and printing their ids to check.

In [7]:
import ipyparallel as parallel

clients = parallel.Client()
clients.block = True  # use synchronous computations
clients.activate()    # enables the %%px magic command
print(clients.ids)

[0, 1, 2]


Importing mpi4py and numpy.

In [8]:
%%px
from mpi4py import MPI
import numpy as np

Implementing Convolution, you don't need to modify this code.

In [9]:
%%px
def convolve_func(main,kernel,KERNEL_DIM,DIMx,DIMy,upper_pad,lower_pad):
	num_pads = int((KERNEL_DIM - 1) / 2)
	conv = np.zeros(main.shape,dtype=int)
	main = np.concatenate((upper_pad,main,lower_pad))
	for i in range(DIMy):
		for j in range(DIMx):
			for k in range(KERNEL_DIM):
				for l in range(KERNEL_DIM):
					if j+l <= DIMx+1 and i+k>=num_pads and i+k<=DIMy:
						conv[j*DIMy+i] += main[(j+l)*DIMy+i-num_pads+k]#*kernel[k][l]
	return conv

5 points: 
Load MPI communicator, get the total number of processes and rank of the process                              

In [10]:
%%px
#and also print total number of processes and rank from each process

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

print(f"Process {rank} out of {size} reporting in.")

[stdout:0] Process 0 out of 3 reporting in.


[stdout:1] Process 1 out of 3 reporting in.


[stdout:2] Process 2 out of 3 reporting in.


5 points: 
Load or initialize data array and kernel array only in process 0(rank 0)                                      

In [12]:
%%px
DIMx = 0
DIMy = 0
KERNEL_DIM = 0
img = None
kernel = None

#Add a condition such that these intializations below should happen in only process 0
if rank == 0:
    img = np.array([[3, 9, 5, 9],[1, 7, 4, 3],[2, 1, 6, 5],[3, 9, 5, 9],[1, 7, 4, 3],[2, 1, 6, 5],[3, 9, 5, 9],[1, 7, 4, 3],[2, 1, 6, 5]])
    kernel = np.array([[0, 1, 0],[0, 0, 0],[0, -1, 0]])
    DIMx = img.shape[0]
    DIMy = img.shape[1]
    KERNEL_DIM = int(kernel.shape[0])

10 points: 
Broadcast data and kernel array sizes from process 0 to  all other processes                                 

In [13]:
%%px
#broadcast data and kernel array sizes (think why we are broadcasting sizes)
DIMx = comm.bcast(DIMx, root=0)
DIMy = comm.bcast(DIMy, root=0)
KERNEL_DIM = comm.bcast(KERNEL_DIM, root=0)

Initialize empty kernel array for all  processes except rank = 0, why we are not initialzing kernel array for rank 0?

Ans: We are not because rank 0 already holds the actual kernel data that we had defined earlier. If we overwrote it with an empty array, we would destroy the data before we had the chance to broadcast it to the other processes.

In [14]:
%%px
#initialize empty kernel array except for process 0(rank=0)
if rank != 0:
    kernel = np.empty((KERNEL_DIM, KERNEL_DIM), dtype=int)

10 points: 
Broadcast Kernel array from rank 0 to all other processes.                                                   

In [15]:
%%px
#broadcast kernel array from rank 0 to all other processes
kernel = comm.bcast(kernel, root=0)

25 points: 
Split the rows in data array equally and scatter them from process 0 to all other process. To split them 
equally, number of rows in the data array must be a integral multiple of number of processes. MPI has ways 
to send unequal chunks of data between processses. But for here you can do with equal number.

In [16]:
%%px
#split and send data array to corresponding processses (you need to initialize a buffer to receive data from 
#process 0, similar to the random initializing done for kernel array)

if rank == 0:
    split_data = np.array_split(img, size, axis=0)
else:
    split_data = None

#Here does we initialize buffer for process 0 also, if so why?(Hint: because of the function we are using to send 
#and receieve data)

local_img = comm.scatter(split_data, root=0)

25 points: 
For convolution of kernel array and data array, you have to pass the kernel padding rows from one
process to another. please see objective for more details. Send and Recieve rows from one process 
to other. Careful with the data size and tags you are sending and receiving should match otherwise
commincator will wait for them indefintely.                                                                  

In [18]:
%%px
#send padding rows from one process to other (carefully observe which process to send data to which process and
# which process receives the data)

upper_pad = np.zeros(DIMy, dtype=int)
lower_pad = np.zeros(DIMy, dtype=int) 

# sending bottom row to next process and receiving upper pad from previous process
if rank < size - 1:
    comm.Send(local_img[-1, :].copy(), dest=rank+1, tag=11)
if rank > 0:
    comm.Recv(upper_pad, source=rank-1, tag=11)

# sending top row to previous process and receiving lower pad from next process
if rank > 0:
    comm.Send(local_img[0, :].copy(), dest=rank-1, tag=22)
if rank < size - 1:
    comm.Recv(lower_pad, source=rank+1, tag=22)


Why we are loading data into process 0 and broadcasting input data to all other processes? are there any other methods to load data into all processes (not for evaluation)

Ans: We are loading the data into a single root process to simulate reading from a unified file system and to prevent I/O access conflicts from mulitple processes trying to read the same file at the identical moment.
Another method is parallel I/O where all processes read their assigned partitions directly from a shared file concurrently without routing it through process 0.

5 points: 
Perform Convolution operation by calling convolve_func() provided for each of the process with 
corresponding rows as arguments.                                                                             

In [19]:
%%px
#convolution function arguments
#main - data array (flattened array), only the part of the data array that is processed for each process
#kernel - kernel array
#DIMy - ColumnSize
#Dimx - RowSize
#upper_pad = upper padding row
#lower_pad = lower padding row

main_flat = local_img.flatten()
upper_pad_flat = upper_pad.flatten()
lower_pad_flat = lower_pad.flatten()
local_DIMx = local_img.shape[0]

local_conv = convolve_func(main_flat, kernel, KERNEL_DIM, local_DIMx, DIMy, upper_pad_flat, lower_pad_flat)

10 points: 
Gather the computed convolutional matrix rows to process 0.                                                 

In [20]:
%%px
#To receive data from all processes, process 0 should have a buffer

gathered_conv = comm.gather(local_conv, root=0)

Reshape the flattened array to match input dimensions

In [21]:
%%px
#Reshape the collected array to the input image dimensions

if rank == 0:
    # concatenating the list of 1D chunks back into a single 1D array
    recvbuf_flat = np.concatenate(gathered_conv)
    # reshaping the flattened array back to the original 2D image dimensions
    recvbuf = np.reshape(recvbuf_flat, (DIMx, DIMy))
else:
    recvbuf = None

5 points: 
Test to check sequential convolution and MPI based parallel convolution outputs                               

In [25]:
%%px
if rank == 0:
    #main_grid is the actual input input image array that is flattened
    #convolution function arguments
    #main_grid - data array (flattened array)
    #kernel - kernel array
    #DIMy - ColumnSize
    #Dimx - RowSize
    #upper_pad = upper padding row
    #lower_pad = lower padding row
    
    #rename the below arguments according to your variable names
    
    # Steps:

    # 1. Flatten the original image for the sequential test
    main_grid = img.flatten()
    
    # 2. Create a row of zeros for the top and bottom padding
    pad_row = np.zeros(DIMy, dtype=int)
    
    # 3. Compute the entire convolution in a single process 
    conv1 = convolve_func(main_grid, kernel, KERNEL_DIM, DIMx, DIMy, pad_row, pad_row)
    conv1 = np.reshape(conv1, (-1, DIMx))
    
    # 4. Check if sequential output matches our MPI parallel output (recvbuf)
    print(np.array_equal(conv1, recvbuf))

[stdout:0] False
